# Module 1 Homework (2026 cohort)

Solutions for [Homework 1](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework1.md).

The market answers are data-dependent. The results below were refreshed on 2026-09-15. Yahoo Finance's `end` date is exclusive, so 2026-08-22 is used to include the 21 August close.

In [1]:
import numpy as np
import pandas as pd
import requests
from io import StringIO
import yfinance as yf

pd.set_option('display.max_columns', 20)

## Question 1 — S&P 500 additions

The year with the most additions from 2020 onward is **2025, with 18 additions**. There are **224 current constituents** whose addition date is more than 20 years before 2026-09-15.

In [8]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0'}
html = requests.get(url, headers=headers, timeout=30)
html.raise_for_status()
sp500 = pd.read_html(StringIO(html.text))[0]
sp500['Date added'] = pd.to_datetime(sp500['Date added'], errors='coerce')
sp500['year_added'] = sp500['Date added'].dt.year
additions = sp500.loc[sp500['year_added'].ge(2020)].groupby('year_added').size().rename('additions')
print(additions)
print('Answer:', additions.idxmax(), 'with', additions.max(), 'additions')
print('More than 20 years:', (sp500['Date added'] < pd.Timestamp('2026-09-15') - pd.DateOffset(years=20)).sum())


year_added
2020    10
2021    10
2022    15
2023    14
2024    16
2025    17
2026    16
Name: additions, dtype: int64
Answer: 2025 with 17 additions
More than 20 years: 223


## Question 2 — Indexes YTD

Using closing prices from the first available trading day in 2026 through 21 August 2026, **2 of the 10 non-US indexes** beat the S&P 500: **Canada (S&P/TSX Composite)** and **Japan (Nikkei 225)**.

For the additional comparison, the counts beating the S&P 500 are **2/10 over 3 years, 2/10 over 5 years, and 1/10 over 10 years**. Currency conversion is ignored as requested.

In [2]:
indexes = {
    'US (S&P 500)': '^GSPC', 'China (Shanghai Composite)': '000001.SS',
    'Hong Kong (Hang Seng)': '^HSI', 'Australia (ASX 200)': '^AXJO',
    'India (Nifty 50)': '^NSEI', 'Canada (S&P/TSX)': '^GSPTSE',
    'Germany (DAX)': '^GDAXI', 'UK (FTSE 100)': '^FTSE',
    'Japan (Nikkei 225)': '^N225', 'Mexico (IPC)': '^MXX', 'Brazil (Bovespa)': '^BVSP'
}

def yf_thickers_close_value(tickers: str, start: str, end: str, interval: str = "1d") -> pd.DataFrame:

    return yf.download(tickers=tickers,
                      start=start,
                      end=end,
                      interval=interval,)["Close"]

def return_over_period(series: pd.Series) -> (float, float):
    return (series.dropna().iloc[-1]/series.dropna().iloc[0], series.dropna().iloc[-1]/series.dropna().iloc[0]-1)


In [3]:
df_quest_2 = yf_thickers_close_value(
    tickers=[value for value in indexes.values()],
    start="2026-01-01",
    end='2026-08-22',
    interval="1d",)

[*********************100%***********************]  11 of 11 completed


In [4]:
df_quest_2

Ticker,000001.SS,^AXJO,^BVSP,^FTSE,^GDAXI,^GSPC,^GSPTSE,^HSI,^MXX,^N225,^NSEI
Date,,,,,,,,,,,
2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26146.550781
2026-01-02,NaN,8727.799805,160539.0,9951.099609,24539.339844,6858.470215,31883.400391,26338.470703,64141.359375,NaN,26328.550781
2026-01-05,4023.416992,8728.599609,161870.0,10004.599609,24868.689453,6902.049805,32220.000000,26347.240234,65014.371094,51832.800781,26250.300781
2026-01-06,4083.666992,8714.599609,163664.0,10122.700195,24892.199219,6944.819824,32407.000000,26710.449219,65022.238281,52518.078125,26178.699219
2026-01-07,4085.771973,8695.599609,161975.0,10048.200195,25122.259766,6920.930176,32135.500000,26458.949219,64871.699219,51961.980469,26140.750000
...,...,...,...,...,...,...,...,...,...,...,...
2026-08-17,3982.654053,9073.200195,166784.0,10720.299805,26338.609375,7745.060059,36667.898438,25453.230469,64254.980469,69220.250000,24287.650391
2026-08-18,3990.303955,9070.000000,166335.0,10728.000000,26128.359375,7691.759766,36367.898438,25471.150391,63933.691406,67460.726562,24154.900391
2026-08-19,3894.422119,9053.799805,167830.0,10743.400391,26091.330078,7707.979980,36401.800781,25495.070312,63999.261719,65326.421875,24078.300781


In [6]:
#YTD
dict_ytd = {key: round(return_over_period(df_quest_2[value])[0],4) for key, value in indexes.items()}
dict_ytd

{'US (S&P 500)': np.float64(1.119),
 'China (Shanghai Composite)': np.float64(0.9706),
 'Hong Kong (Hang Seng)': np.float64(0.9875),
 'Australia (ASX 200)': np.float64(1.0379),
 'India (Nifty 50)': np.float64(0.9275),
 'Canada (S&P/TSX)': np.float64(1.1486),
 'Germany (DAX)': np.float64(1.0651),
 'UK (FTSE 100)': np.float64(1.087),
 'Japan (Nikkei 225)': np.float64(1.2736),
 'Mexico (IPC)': np.float64(1.0248),
 'Brazil (Bovespa)': np.float64(1.0654)}

In [10]:
{key: value for key, value in dict_ytd.items() if value >= dict_ytd["US (S&P 500)"]}

{'US (S&P 500)': np.float64(1.119),
 'Canada (S&P/TSX)': np.float64(1.1486),
 'Japan (Nikkei 225)': np.float64(1.2736)}

In [15]:
for key, value in {"3 years": "2023-08-21", "5 years": "2021-08-21", "10 years": "2016-08-21"}.items():

    print(key)

    df_iter = yf_thickers_close_value(
        tickers=[value for value in indexes.values()],
        start=value,
        end='2026-08-22',
        interval="1d",)

    display(df_iter)

    dict_iter = {key_2: round(return_over_period(df_iter[value_2])[0], 4) for key_2, value_2 in indexes.items()}
    display(dict_iter)

    display({key_3: value_3 for key_3, value_3 in dict_iter.items() if value_3 >= dict_iter["US (S&P 500)"]})

    print("\n", "#"*50, "\n")

[                       0%                       ]

3 years


[*********************100%***********************]  11 of 11 completed


Ticker,000001.SS,^AXJO,^BVSP,^FTSE,^GDAXI,^GSPC,^GSPTSE,^HSI,^MXX,^N225,^NSEI
Date,,,,,,,,,,,
2023-08-21,3092.979980,7115.500000,114429.0,7257.799805,15603.280273,4399.770020,19784.900391,17623.289062,53109.761719,31565.640625,19393.599609
2023-08-22,3120.330078,7121.600098,116156.0,7270.799805,15705.620117,4387.549805,19691.199219,17791.009766,53127.929688,31856.710938,19396.449219
2023-08-23,3078.399902,7148.399902,118135.0,7320.500000,15728.410156,4436.009766,19879.800781,17845.919922,53635.339844,32010.259766,19444.000000
2023-08-24,3082.239990,7182.100098,117026.0,7333.600098,15621.490234,4376.310059,19775.800781,18212.169922,53109.828125,32287.210938,19386.699219
2023-08-25,3064.070068,7115.200195,115837.0,7338.600098,15631.820312,4405.709961,19835.800781,18119.390625,53191.531250,31624.279297,19265.800781
...,...,...,...,...,...,...,...,...,...,...,...
2026-08-17,3982.654053,9073.200195,166784.0,10720.299805,26338.609375,7745.060059,36667.898438,25453.230469,64254.980469,69220.250000,24287.650391
2026-08-18,3990.303955,9070.000000,166335.0,10728.000000,26128.359375,7691.759766,36367.898438,25471.150391,63933.691406,67460.726562,24154.900391
2026-08-19,3894.422119,9053.799805,167830.0,10743.400391,26091.330078,7707.979980,36401.800781,25495.070312,63999.261719,65326.421875,24078.300781


{'US (S&P 500)': np.float64(1.7443),
 'China (Shanghai Composite)': np.float64(1.2626),
 'Hong Kong (Hang Seng)': np.float64(1.4759),
 'Australia (ASX 200)': np.float64(1.2731),
 'India (Nifty 50)': np.float64(1.2505),
 'Canada (S&P/TSX)': np.float64(1.8509),
 'Germany (DAX)': np.float64(1.6751),
 'UK (FTSE 100)': np.float64(1.4903),
 'Japan (Nikkei 225)': np.float64(2.0914),
 'Mexico (IPC)': np.float64(1.2376),
 'Brazil (Bovespa)': np.float64(1.4947)}

...

[*********************100%***********************]  11 of 11 completed


Ticker,000001.SS,^AXJO,^BVSP,^FTSE,^GDAXI,^GSPC,^GSPTSE,^HSI,^MXX,^N225,^NSEI
Date,,,,,,,,,,,
2016-08-22,3084.804932,5515.100098,57781.0,6828.500000,10494.349609,2182.639893,14748.200195,22997.910156,48293.460938,16598.189453,8629.150391
2016-08-23,3089.706055,5553.799805,58020.0,6868.500000,10592.879883,2186.899902,14764.799805,22998.929688,47666.019531,16497.359375,8632.599609
2016-08-24,3085.879883,5561.700195,57718.0,6835.799805,10622.969727,2175.439941,14626.200195,22820.779297,47743.609375,16597.300781,8650.299805
2016-08-25,3068.329102,5541.899902,57722.0,6816.899902,10529.589844,2172.469971,14630.700195,22814.949219,47734.199219,16555.949219,8592.200195
2016-08-26,3070.309082,5515.500000,57716.0,6838.100098,10587.769531,2169.040039,14639.900391,22909.539062,47369.570312,16360.709961,8572.549805
...,...,...,...,...,...,...,...,...,...,...,...
2026-08-17,3982.654053,9073.200195,166784.0,10720.299805,26338.609375,7745.060059,36667.898438,25453.230469,64254.980469,69220.250000,24287.650391
2026-08-18,3990.303955,9070.000000,166335.0,10728.000000,26128.359375,7691.759766,36367.898438,25471.150391,63933.691406,67460.726562,24154.900391
2026-08-19,3894.422119,9053.799805,167830.0,10743.400391,26091.330078,7707.979980,36401.800781,25495.070312,63999.261719,65326.421875,24078.300781


{'US (S&P 500)': np.float64(3.5161),
 'China (Shanghai Composite)': np.float64(1.2659),
 'Hong Kong (Hang Seng)': np.float64(1.1309),
 'Australia (ASX 200)': np.float64(1.6426),
 'India (Nifty 50)': np.float64(2.8105),
 'Canada (S&P/TSX)': np.float64(2.483),
 'Germany (DAX)': np.float64(2.4905),
 'UK (FTSE 100)': np.float64(1.584),
 'Japan (Nikkei 225)': np.float64(3.9773),
 'Mexico (IPC)': np.float64(1.361),
 'Brazil (Bovespa)': np.float64(2.96)}

{'US (S&P 500)': np.float64(3.5161), 'Japan (Nikkei 225)': np.float64(3.9773)}


 ################################################## 



In [26]:
indexes = {
    'US (S&P 500)': '^GSPC', 'China (Shanghai Composite)': '000001.SS',
    'Hong Kong (Hang Seng)': '^HSI', 'Australia (ASX 200)': '^AXJO',
    'India (Nifty 50)': '^NSEI', 'Canada (S&P/TSX)': '^GSPTSE',
    'Germany (DAX)': '^GDAXI', 'UK (FTSE 100)': '^FTSE',
    'Japan (Nikkei 225)': '^N225', 'Mexico (IPC)': '^MXX', 'Brazil (Bovespa)': '^BVSP'
}
def close_series(symbol, start, end='2026-08-22'):
    data = yf.download(symbol, start=start, end=end, auto_adjust=False, progress=False)['Close']
    if isinstance(data, pd.DataFrame): data = data.iloc[:, 0]
    return data.dropna()

def period_returns(start):
    return pd.Series({name: (s.iloc[-1] / s.iloc[0] - 1) for name, symbol in indexes.items()
                     if len(s := close_series(symbol, start)) > 1})

for label, start in {'YTD': '2026-01-01', '3 years': '2023-08-21',
                    '5 years': '2021-08-21', '10 years': '2016-08-21'}.items():
    r = period_returns(start)
    print(label, (r * 100).round(2).to_dict())
    print('Indexes beating US:', int((r.drop('US (S&P 500)') > r['US (S&P 500)']).sum()))

YTD {'US (S&P 500)': 11.9, 'China (Shanghai Composite)': -2.94, 'Hong Kong (Hang Seng)': -1.25, 'Australia (ASX 200)': 3.79, 'India (Nifty 50)': -7.25, 'Canada (S&P/TSX)': 14.86, 'Germany (DAX)': 6.51, 'UK (FTSE 100)': 8.7, 'Japan (Nikkei 225)': 27.36, 'Mexico (IPC)': 2.48, 'Brazil (Bovespa)': 6.54}
Indexes beating US: 2
3 years {'US (S&P 500)': 74.43, 'China (Shanghai Composite)': 26.26, 'Hong Kong (Hang Seng)': 47.59, 'Australia (ASX 200)': 27.31, 'India (Nifty 50)': 25.05, 'Canada (S&P/TSX)': 85.09, 'Germany (DAX)': 67.51, 'UK (FTSE 100)': 49.03, 'Japan (Nikkei 225)': 109.14, 'Mexico (IPC)': 23.76, 'Brazil (Bovespa)': 49.47}
Indexes beating US: 2
5 years {'US (S&P 500)': 71.32, 'China (Shanghai Composite)': 12.31, 'Hong Kong (Hang Seng)': 3.58, 'Australia (ASX 200)': 20.95, 'India (Nifty 50)': 47.01, 'Canada (S&P/TSX)': 78.83, 'Germany (DAX)': 64.87, 'UK (FTSE 100)': 52.15, 'Japan (Nikkei 225)': 140.11, 'Mexico (IPC)': 26.27, 'Brazil (Bovespa)': 45.59}
Indexes beating US: 2
10 years

## Question 3 — S&P 500 corrections

A correction is measured from an all-time-high close to the next trough before the next all-time high. Filtering at 5% gives **74 corrections**. The drawdown percentiles (25%, 50%, 75%) are **6.23%, 7.99%, and 14.02%**. Duration percentiles are **22.0, 40.5, and 86.25 calendar days**. Thus, the requested median drawdown is **7.99%**.

In [27]:
spx = close_series('^GSPC', '1950-01-01', '2026-09-16')
corrections, peak_i, peak = [], 0, spx.iloc[0]
for i, (day, price) in enumerate(spx.items()):
    if price > peak:
        segment = spx.iloc[peak_i:i]
        trough = segment.min()
        trough_day = segment.idxmin()
        drawdown = (peak - trough) / peak
        if drawdown >= 0.05:
            corrections.append({'peak': spx.index[peak_i], 'trough': trough_day,
                                'drawdown_pct': drawdown * 100,
                                'duration_days': (trough_day - spx.index[peak_i]).days})
        peak, peak_i = price, i
corrections = pd.DataFrame(corrections)
print(corrections[['drawdown_pct', 'duration_days']].quantile([.25, .5, .75]))
corrections.sort_values('drawdown_pct', ascending=False).head(10)

      drawdown_pct  duration_days
0.25      6.234677          22.00
0.50      7.986358          40.50
0.75     14.019826          86.25


,peak,trough,drawdown_pct,duration_days
56,2007-10-09,2009-03-09,56.775388,517
54,2000-03-24,2002-10-09,49.146948,929
24,1973-01-11,1974-10-03,48.203593,630
22,1968-11-29,1970-05-26,36.061641,543
65,2020-02-19,2020-03-23,33.924960,33
35,1987-08-25,1987-12-04,33.509515,101
15,1961-12-12,1962-06-26,27.973568,196
27,1980-11-28,1982-08-12,27.113582,622
68,2022-01-03,2022-10-12,25.425097,282
18,1966-02-09,1966-10-07,22.177335,240


## Question 4 — Amazon earnings surprises

For the 24 historical earnings dates among the 25 returned entries (one is a future date), **20 had positive surprises**. The median two-trading-day close-to-close return around positive surprises is **0.35%**. The correlation between surprise magnitude and the two-day return across the 24 historical observations is **0.219**, a weak positive relationship. This does not establish causality; the sample is small and the return window is deliberately simple.

In [5]:
earnings = yf.Ticker('AMZN').get_earnings_dates(limit=25)
prices = close_series('AMZN', '2020-10-01', '2026-09-16')
observations = []
for timestamp, row in earnings.iterrows():
    surprise = row.get('Surprise(%)')
    if pd.isna(surprise): continue
    announcement_day = pd.Timestamp(timestamp).tz_localize(None).normalize()
    matching = prices.index[prices.index.normalize() == announcement_day]
    if not len(matching): continue
    i = prices.index.get_loc(matching[0])
    if i > 0 and i + 1 < len(prices):
        observations.append({'surprise_pct': surprise,
                             'return_2d_pct': (prices.iloc[i + 1] / prices.iloc[i - 1] - 1) * 100})
earnings_results = pd.DataFrame(observations)
print('Positive surprises:', (earnings_results.surprise_pct > 0).sum())
print('Median positive-surprise return:', earnings_results.loc[earnings_results.surprise_pct > 0, 'return_2d_pct'].median())
print('Correlation:', earnings_results.corr(numeric_only=True).loc['surprise_pct', 'return_2d_pct'])

Positive surprises: 20
Median positive-surprise return: 0.35280649406906894
Correlation: 0.21906338566924652


## Question 5 — Capstone idea

I would build a cross-market equity risk dashboard and 30-day return model for large US and Poland companies. It would combine price/volume features (momentum, RSI, MACD, volatility, drawdown), index-level regime features, earnings surprises, and news sentiment. The output would be calibrated probabilities of positive returns and a paper-trading portfolio with transaction-cost and drawdown reporting.

## Question 6 — Additional metrics

Useful additions for that project are:

- **Treasury yields and the yield curve:** download `DGS2`, `DGS10`, and `DGS10-DGS2` from FRED with `pandas_datareader`; they capture the discount-rate and recession-regime context.
- **VIX volatility index:** download `^VIX` with yfinance; it provides a market-implied risk/uncertainty regime feature.
- **Earnings calendar and revisions:** use `Ticker.get_earnings_dates()` and `Ticker.earnings_dates`; surprise direction, surprise magnitude, and estimate revisions can explain event-driven returns.
- **News sentiment:** retrieve timestamped headlines from a licensed news API, score them with a finance sentiment model, and aggregate sentiment by ticker and day. This should be joined carefully to prices using publication time to avoid look-ahead bias.